# Reproducible analysis code — inflammatory biomarkers before and after the COVID-19 pandemic

This notebook contains only the analyses reported in the manuscript and supplementary material submitted to the *International Journal of Epidemiology*.

**Input data are not included** because they contain individual-level participant information. Place the analytical dataset at `Data/11_06_biomarker_metafile_for_python.csv` or edit `DATA_PATH` below.

Main outputs reproduced here:
- Table 1: participant characteristics
- Table S1: crude biomarker distributions
- Figure 1: semester-specific biomarker trajectories
- Figure 2: adjusted pre/post-pandemic differences
- Table S2: sequential adjusted models
- Table S3: year-specific adjusted differences
- Table S4: principal component analysis
- Table S5: lifestyle–biomarker associations
- Table S6 and Figure 3: lifestyle-stratified pre/post-pandemic differences

## 1. Setup and common data preparation

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from scipy.stats import mannwhitneyu, chi2_contingency, kruskal
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
DATA_PATH = ROOT / "Data" / "analysis_dataset.csv"
RESULTS_DIR = ROOT / "Data" / "results"
FIGURES_DIR = ROOT / "Data" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Analytical dataset not found: {DATA_PATH}\n"
        "Place the restricted dataset in Data/ or edit DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

# Core biomarker order used throughout the manuscript
biomarkers = [
    "CRP mg/l",
    "MCP-1 pg/mL",
    "TNF-α pg/mL",
    "IFNγ pg/mL",
    "Eotaxin pg/mL",
    "IL-6 pg/mL",
    "IL-8 pg/mL",
    "IL-10 pg/mL",
    "sICAM-1 (pg/mL)",
    "sVCAM-1 (pg/mL)",
    "VEGF-A pg/mL",
    "MMP-9 (ng/mL)",
]
year_order = [2019, 2023, 2024]

# Standardize core variables once
if "age" not in df.columns and "age.x" in df.columns:
    df["age"] = df["age.x"]
df["age"] = pd.to_numeric(df["age"], errors="coerce")

rename_map = {
    "Moderate PA, standardized data, PAFQ (min/day)": "PAFQ_MPA_raw",
    "Vigorous PA, standardized data, PAFQ (min/day)": "PAFQ_VPA_raw",
    "Sedentary, standardized data, PAFQ (min/day)": "PAFQ_SE_raw",
}
for old, new in rename_map.items():
    if old in df.columns and new not in df.columns:
        df = df.rename(columns={old: new})

# Harmonize period if necessary
if "period" not in df.columns:
    y = pd.to_numeric(df["year"], errors="coerce")
    df["period"] = np.where(y.eq(2019), "pre_pandemic",
                    np.where(y.isin([2023, 2024]), "post_pandemic", np.nan))
else:
    df["period"] = df["period"].astype(str).replace({
        "Pre-pandemic": "pre_pandemic", "Post-pandemic": "post_pandemic",
        "pre": "pre_pandemic", "post": "post_pandemic",
        "2019": "pre_pandemic", "2023": "post_pandemic", "2024": "post_pandemic",
    })
    df.loc[~df["period"].isin(["pre_pandemic", "post_pandemic"]), "period"] = np.nan

# Education, income and smoking: retain missing values as explicit model categories
if "education_3cat" not in df.columns:
    df["education_3cat"] = df["m_education_coded"].replace({
        "Primary": "Primary", "Secondary": "Secondary", "Tertiary": "Tertiary", "Autre": np.nan
    })
if "income_3cat" not in df.columns:
    df["income_3cat"] = df["m_income_grp"].replace({
        "Faible": "Low", "Modéré faible": "Low", "Modéré élevé": "Middle",
        "Elevé": "High", "Inconnu": np.nan, "Refus": np.nan
    })

df["education_3cat_model"] = df["education_3cat"].fillna("Missing")
df["income_3cat_model"] = df["income_3cat"].fillna("Missing")
df["smoking_status_model"] = df["smoking_status"].fillna("Missing")

for v in ["bmi_nursemeasured", "PAFQ_MPA_raw", "PAFQ_VPA_raw", "PAFQ_SE_raw", "somttot", "sumalco"]:
    if v in df.columns:
        df[v] = pd.to_numeric(df[v], errors="coerce")

if "PAFQ_MPA_raw" in df.columns and "PAFQ_VPA_raw" in df.columns:
    df["physical_activity_min_day"] = df["PAFQ_MPA_raw"] + df["PAFQ_VPA_raw"]

print(f"Loaded {len(df)} participants from {DATA_PATH}")

## 2. Table 1 — Participant characteristics by pandemic period

In [ ]:
# ============================================================
# Table 1 — Participant characteristics by pandemic period
# Including variables used in main model + Model 4 sensitivity
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency
from pathlib import Path

# ------------------------------------------------------------
# 1. Basic setup
# ------------------------------------------------------------

# Make sure age exists
if "age" not in df.columns and "age.x" in df.columns:
    df["age"] = df["age.x"]

df["age"] = pd.to_numeric(df["age"], errors="coerce")

# Rename lifestyle variables safely
df = df.rename(columns={
    "Moderate PA, standardized data, PAFQ (min/day)": "PAFQ_MPA_raw",
    "Vigorous PA, standardized data, PAFQ (min/day)": "PAFQ_VPA_raw",
    "Sedentary, standardized data, PAFQ (min/day)": "PAFQ_SE_raw"
})

# Recode education into 3 categories
df["education_3cat"] = df["m_education_coded"].replace({
    "Primary": "Primary",
    "Secondary": "Secondary",
    "Tertiary": "Tertiary",
    "Autre": np.nan
})

# Recode income into 3 categories
df["income_3cat"] = df["m_income_grp"].replace({
    "Faible": "Low",
    "Modéré faible": "Low",
    "Modéré élevé": "Middle",
    "Elevé": "High",
    "Inconnu": np.nan,
    "Refus": np.nan
})

# Harmonize variables for descriptive table
df["education_3cat_table"] = df["education_3cat"].fillna("Missing")
df["income_3cat_table"] = df["income_3cat"].fillna("Missing")
df["smoking_status_table"] = df["smoking_status"].fillna("Missing")
df["season_table"] = df["season"].fillna("Missing")

# Vegetarian status
if "vegetarian" in df.columns:
    df["vegetarian_table"] = df["vegetarian"].replace({
        0: "No",
        1: "Yes",
        "0": "No",
        "1": "Yes",
        "No": "No",
        "Yes": "Yes"
    }).fillna("Missing")

# Convert Model 4 continuous variables to numeric
# Create total physical activity variable: moderate + vigorous PA
for v in ["PAFQ_MPA_raw", "PAFQ_VPA_raw", "PAFQ_SE_raw", "somttot", "sumalco"]:
    if v in df.columns:
        df[v] = pd.to_numeric(df[v], errors="coerce")

df["physical_activity_min_day"] = df["PAFQ_MPA_raw"] + df["PAFQ_VPA_raw"]

# Convert final Model 4 continuous variables to numeric
model4_numeric_vars = [
    "physical_activity_min_day",
    "PAFQ_SE_raw",
    "somttot",
    "sumalco"
]

for v in model4_numeric_vars:
    if v in df.columns:
        df[v] = pd.to_numeric(df[v], errors="coerce")

# Define period order
group_col = "period"
group_order = ["pre_pandemic", "post_pandemic"]

df_table1 = df[df[group_col].isin(group_order)].copy()

# ------------------------------------------------------------
# 2. Variables for Table 1
# ------------------------------------------------------------

continuous_vars = {
    "age": "Age, years",
    "bmi_nursemeasured": "Measured BMI, kg/m²",
    "physical_activity_min_day": "Total physical activity, min/day",
    "PAFQ_SE_raw": "Sedentary time, min/day",
    "somttot": "Total sleep duration",
    "sumalco": "Alcohol intake, g/day"
}

# Keep only continuous variables that exist
continuous_vars = {
    var: label for var, label in continuous_vars.items()
    if var in df_table1.columns
}

categorical_vars = {
    "age_group_new": "Age group",
    "sex_en": "Sex",
    "education_3cat_table": "Education level",
    "income_3cat_table": "Income category",
    "smoking_status_table": "Smoking status",
    "bmi_category": "BMI category",
    "season_table": "Season of blood sampling",
    "self_reported_and_detected_diabetes": "Diabetes",
    "self_reported_and_detected_HTA": "Hypertension",
    "self_reported_and_detected_high_cholesterol": "High cholesterol",
}

# Keep only categorical variables that exist
categorical_vars = {
    var: label for var, label in categorical_vars.items()
    if var in df_table1.columns
}

# Preferred category orders
category_orders = {
    "age_group_new": ["35-44", "45-54", "55-64", "65-75", "Missing"],
    "sex_en": ["Female", "Male", "Missing"],
    "education_3cat_table": ["Primary", "Secondary", "Tertiary", "Missing"],
    "income_3cat_table": ["Low", "Middle", "High", "Missing"],
    "smoking_status_table": ["Never smoker", "Ex-smoker", "Current smoker", "Missing"],
    "bmi_category": ["Underweight", "Normal weight", "Overweight", "Obese", "Missing"],
    "season_table": ["Spring", "Summer", "Fall", "Winter", "Missing"],
    "self_reported_and_detected_diabetes": ["No", "Yes", "Missing"],
    "self_reported_and_detected_HTA": ["No", "Yes", "Missing"],
    "self_reported_and_detected_high_cholesterol": ["No", "Yes", "Missing"],
}

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def format_pvalue(p):
    if pd.isna(p):
        return "—"
    elif p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"


def median_iqr(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    q1 = x.quantile(0.25)
    med = x.median()
    q3 = x.quantile(0.75)
    return f"{med:.1f} [{q1:.1f}, {q3:.1f}]"


def n_pct(n, denom):
    if denom == 0:
        return "0 (0.0)"
    return f"{int(n)} ({n / denom * 100:.1f})"


def get_categories(tmp_display, var):
    observed = tmp_display[var].dropna().astype(str).unique().tolist()
    
    if var in category_orders:
        ordered = [c for c in category_orders[var] if c in observed]
        extra = sorted([c for c in observed if c not in ordered])
        return ordered + extra
    
    return sorted(observed)

# ------------------------------------------------------------
# 4. Build Table 1
# ------------------------------------------------------------

rows = []

total_n = len(df_table1)

group_counts = (
    df_table1[group_col]
    .value_counts()
    .reindex(group_order)
    .fillna(0)
    .astype(int)
)

# Header row
header = {
    "Variable": "",
    "Category": f"Total (n={total_n})",
    "Total": f"{total_n} (100.0)",
    "p-value": ""
}

for g in group_order:
    n = group_counts[g]
    header[g] = n_pct(n, total_n)

rows.append(header)

# ------------------------------------------------------------
# Continuous variables
# ------------------------------------------------------------

for var, label in continuous_vars.items():
    
    row = {
        "Variable": label,
        "Category": "Median [IQR]",
        "Total": median_iqr(df_table1[var])
    }
    
    for g in group_order:
        row[g] = median_iqr(df_table1.loc[df_table1[group_col] == g, var])
    
    x1 = pd.to_numeric(
        df_table1.loc[df_table1[group_col] == group_order[0], var],
        errors="coerce"
    ).dropna()
    
    x2 = pd.to_numeric(
        df_table1.loc[df_table1[group_col] == group_order[1], var],
        errors="coerce"
    ).dropna()
    
    if len(x1) > 0 and len(x2) > 0:
        _, p = mannwhitneyu(x1, x2, alternative="two-sided")
    else:
        p = np.nan
    
    row["p-value"] = format_pvalue(p)
    rows.append(row)

# ------------------------------------------------------------
# Categorical variables
# ------------------------------------------------------------

for var, label in categorical_vars.items():
    
    # Header row for variable
    header_idx = len(rows)
    rows.append({
        "Variable": label,
        "Category": "",
        "Total": "",
        group_order[0]: "",
        group_order[1]: "",
        "p-value": ""
    })
    
    tmp = df_table1[[var, group_col]].copy()
    
    # For display, include Missing
    tmp_display = tmp.copy()
    tmp_display[var] = tmp_display[var].fillna("Missing").astype(str)
    
    # For p-value, exclude Missing
    tmp_for_test = tmp_display[tmp_display[var] != "Missing"].copy()
    
    ct = pd.crosstab(tmp_for_test[var], tmp_for_test[group_col])
    ct = ct.reindex(columns=group_order, fill_value=0)
    
    if ct.shape[0] >= 2 and ct.shape[1] >= 2 and ct.to_numpy().sum() > 0:
        try:
            _, p, _, _ = chi2_contingency(ct)
        except Exception:
            p = np.nan
    else:
        p = np.nan
    
    rows[header_idx]["p-value"] = format_pvalue(p)
    
    categories = get_categories(tmp_display, var)
    
    total_denom = len(tmp_display)
    
    for category in categories:
        
        line = {
            "Variable": label,
            "Category": category
        }
        
        n_total_cat = (tmp_display[var] == category).sum()
        line["Total"] = n_pct(n_total_cat, total_denom)
        
        for g in group_order:
            sub = tmp_display[tmp_display[group_col] == g]
            denom = len(sub)
            n_cat = (sub[var] == category).sum()
            line[g] = n_pct(n_cat, denom)
        
        line["p-value"] = ""
        rows.append(line)

# ------------------------------------------------------------
# 5. Final table and save
# ------------------------------------------------------------

table1 = pd.DataFrame(rows)

# Rename period columns for clean display
table1 = table1.rename(columns={
    "pre_pandemic": f"Pre-pandemic (n={group_counts['pre_pandemic']})",
    "post_pandemic": f"Post-pandemic (n={group_counts['post_pandemic']})"
})

# Display
display(table1)

# Save
output_path = RESULTS_DIR / "Table1_participant_characteristics_with_model4_variables.xlsx"

table1.to_excel(output_path, index=False)

print(f"Table 1 saved here:\n{output_path}")

## 3. Table S1 — Crude biomarker distributions by period and year

In [ ]:
# ------------------------------------------------------------
# Helper function for crude percentage change from medians
# ------------------------------------------------------------

def crude_pct_change(reference_values, comparison_values):
    """
    Calculates crude percentage change based on medians:
    100 * (median_comparison / median_reference - 1)
    """
    reference_values = pd.to_numeric(reference_values, errors="coerce").dropna()
    comparison_values = pd.to_numeric(comparison_values, errors="coerce").dropna()
    
    if len(reference_values) == 0 or len(comparison_values) == 0:
        return np.nan
    
    ref_median = reference_values.median()
    comp_median = comparison_values.median()
    
    if ref_median == 0:
        return np.nan
    
    return 100 * (comp_median / ref_median - 1)


def format_pct(x):
    if pd.isna(x):
        return "—"
    elif abs(x) < 0.05:
        return "0.0%"
    else:
        return f"{x:+.1f}%"

In [ ]:
# ============================================================
# Table 2 — Biomarker distributions by period and year
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, kruskal

# ------------------------------------------------------------
# 1. Biomarker list with display units
# ------------------------------------------------------------

biomarker_specs = [
    {"col": "CRP mg/l",          "label": "CRP mg/L",          "factor": 1},
    {"col": "MCP-1 pg/mL",       "label": "MCP-1 pg/mL",       "factor": 1},
    {"col": "TNF-α pg/mL",       "label": "TNF-α pg/mL",       "factor": 1},
    {"col": "IFNγ pg/mL",        "label": "IFNγ pg/mL",        "factor": 1},
    {"col": "Eotaxin pg/mL",     "label": "eotaxin pg/mL",     "factor": 1},
    {"col": "IL-6 pg/mL",        "label": "IL-6 pg/mL",        "factor": 1},
    {"col": "IL-8 pg/mL",        "label": "IL-8 pg/mL",        "factor": 1},
    {"col": "IL-10 pg/mL",       "label": "IL-10 pg/mL",       "factor": 1},
    {"col": "sICAM-1 (pg/mL)",   "label": "sICAM-1 ng/mL",     "factor": 1/1000},
    {"col": "sVCAM-1 (pg/mL)",   "label": "sVCAM-1 ng/mL",     "factor": 1/1000},
    {"col": "VEGF-A pg/mL",      "label": "VEGF-A pg/mL",      "factor": 1},
    {"col": "MMP-9 (ng/mL)",     "label": "MMP-9 ng/mL",       "factor": 1},
]

period_order = ["pre_pandemic", "post_pandemic"]

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def format_pvalue(p):
    if pd.isna(p):
        return "—"
    elif p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"


def median_iqr(x, digits=2):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    q1 = x.quantile(0.25)
    med = x.median()
    q3 = x.quantile(0.75)
    return f"{med:.{digits}f} [{q1:.{digits}f}, {q3:.{digits}f}]"


# ------------------------------------------------------------
# 3. Build Table 2 — period comparison
# ------------------------------------------------------------

# ------------------------------------------------------------
# 3. Build Table 2 — period comparison
# ------------------------------------------------------------

rows_period = []

for spec in biomarker_specs:
    
    biomarker = spec["col"]
    label = spec["label"]
    factor = spec["factor"]
    
    tmp = df[["period", biomarker]].copy()
    tmp[biomarker] = pd.to_numeric(tmp[biomarker], errors="coerce") * factor
    tmp = tmp.dropna(subset=["period", biomarker])
    
    row = {
        "biomarker": label,
        "Total": median_iqr(tmp[biomarker]),
        "pre_pandemic": median_iqr(tmp.loc[tmp["period"] == "pre_pandemic", biomarker]),
        "post_pandemic": median_iqr(tmp.loc[tmp["period"] == "post_pandemic", biomarker]),
    }
    
    x_pre = tmp.loc[tmp["period"] == "pre_pandemic", biomarker].dropna()
    x_post = tmp.loc[tmp["period"] == "post_pandemic", biomarker].dropna()

    row["Post vs pre, %"] = format_pct(crude_pct_change(x_pre, x_post))
    
    if len(x_pre) > 0 and len(x_post) > 0:
        _, p = mannwhitneyu(x_pre, x_post, alternative="two-sided")
    else:
        p = np.nan
    
    row["p_period"] = format_pvalue(p)
    rows_period.append(row)

table2_period = pd.DataFrame(rows_period)

table2_period

In [ ]:
rows_year = []

for spec in biomarker_specs:
    
    biomarker = spec["col"]
    label = spec["label"]
    factor = spec["factor"]
    
    tmp = df[["year", biomarker]].copy()
    tmp[biomarker] = pd.to_numeric(tmp[biomarker], errors="coerce") * factor
    tmp = tmp.dropna(subset=["year", biomarker])
    
    row = {
        "biomarker": label,
        "Total": median_iqr(tmp[biomarker])
    }
    
    groups_for_test = []
    
    for year in year_order:
        values = tmp.loc[tmp["year"] == year, biomarker].dropna()
        row[str(year)] = median_iqr(values)
        
        if len(values) > 0:
            groups_for_test.append(values)
    
    values_2019 = tmp.loc[tmp["year"] == 2019, biomarker].dropna()
    values_2023 = tmp.loc[tmp["year"] == 2023, biomarker].dropna()
    values_2024 = tmp.loc[tmp["year"] == 2024, biomarker].dropna()
    
    row["2023 vs 2019, %"] = format_pct(crude_pct_change(values_2019, values_2023))
    row["2024 vs 2019, %"] = format_pct(crude_pct_change(values_2019, values_2024))
    
    if len(groups_for_test) >= 2:
        _, p = kruskal(*groups_for_test)
    else:
        p = np.nan
    
    row["p_year"] = format_pvalue(p)
    rows_year.append(row)

table2_year = pd.DataFrame(rows_year)
table2_year

In [ ]:
with pd.ExcelWriter(RESULTS_DIR / "Table_S1_biomarker_distributions.xlsx") as writer:
    table2_period.to_excel(writer, sheet_name="Period", index=False)
    table2_year.to_excel(writer, sheet_name="Year", index=False)
print("Saved:", RESULTS_DIR / "Table_S1_biomarker_distributions.xlsx")

## 4. Figure 1 — Semester-specific trajectories of inflammatory biomarkers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.nonparametric.smoothers_lowess import lowess
import warnings

warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)

# ============================================================
# 1. Préparation
# ============================================================

df_fig = df.copy()

# Garder uniquement les périodes d'intérêt
df_fig = df_fig[df_fig["period"].isin(["pre_pandemic", "post_pandemic"])].copy()

# Créer semester si nécessaire
# Option A : semester existe déjà
if "semester" in df_fig.columns:
    df_fig["semester"] = df_fig["semester"].astype(str)

# Option B : sinon, créer semester depuis quarter, ex: 2023-Q3 -> 2023-S2
elif "quarter" in df_fig.columns:
    df_fig["year_tmp"] = df_fig["quarter"].astype(str).str.extract(r"(\d{4})").astype(float)
    df_fig["quarter_tmp"] = df_fig["quarter"].astype(str).str.extract(r"Q(\d)").astype(float)
    df_fig["semester"] = np.where(
        df_fig["quarter_tmp"].isin([1, 2]),
        df_fig["year_tmp"].astype("Int64").astype(str) + "-S1",
        df_fig["year_tmp"].astype("Int64").astype(str) + "-S2"
    )

else:
    raise ValueError("Il faut une colonne 'semester' ou 'quarter' pour créer la figure.")

# Ordre des semestres
ORDER = ["2019-S1", "2019-S2", "2023-S1", "2023-S2", "2024-S1", "2024-S2"]

df_fig = df_fig[df_fig["semester"].isin(ORDER)].copy()

xpos = {s: i for i, s in enumerate(ORDER)}

# ============================================================
# 2. Tes 12 biomarqueurs
# ============================================================

BIOMARKERS = [
    "CRP mg/l",
    "MCP-1 pg/mL",
    "TNF-α pg/mL",
    "IFNγ pg/mL",
    "Eotaxin pg/mL",
    "IL-6 pg/mL",
    "IL-8 pg/mL",
    "IL-10 pg/mL",
    "sICAM-1 (pg/mL)",
    "sVCAM-1 (pg/mL)",
    "VEGF-A pg/mL",
    "MMP-9 (ng/mL)"
]

LABELS = {
    "CRP mg/l": "CRP",
    "MCP-1 pg/mL": "MCP-1",
    "TNF-α pg/mL": "TNF-α",
    "IFNγ pg/mL": "IFNγ",
    "Eotaxin pg/mL": "Eotaxin",
    "IL-6 pg/mL": "IL-6",
    "IL-8 pg/mL": "IL-8",
    "IL-10 pg/mL": "IL-10",
    "sICAM-1 (pg/mL)": "sICAM-1",
    "sVCAM-1 (pg/mL)": "sVCAM-1",
    "VEGF-A pg/mL": "VEGF-A",
    "MMP-9 (ng/mL)": "MMP-9"
}

# Vérifier colonnes manquantes
missing = [b for b in BIOMARKERS if b not in df_fig.columns]
if missing:
    raise ValueError(f"Biomarker columns missing: {missing}")

# Couleurs, une par biomarqueur
colors = plt.cm.tab20(np.linspace(0, 1, len(BIOMARKERS)))
COLOR_MAP = dict(zip(BIOMARKERS, colors))

# Log-transform comme dans le code proposé
for col in BIOMARKERS:
    df_fig[col] = pd.to_numeric(df_fig[col], errors="coerce")
    df_fig[col + "_log"] = np.log(df_fig[col] + 0.001)

# ============================================================
# 3. Vérifier les n par semestre
# ============================================================

n_semester = (
    df_fig
    .groupby("semester")
    .size()
    .reindex(ORDER)
    .reset_index(name="n")
)

display(n_semester)

# ============================================================
# 4. Fonctions LOESS
# ============================================================

def loess_fit(x, y, frac=0.7):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]

    if len(x) < 5:
        return None, None

    out = lowess(y, x, frac=frac, return_sorted=True)
    return out[:, 0], out[:, 1]


def loess_ci(x, y, frac=0.7, n_boot=200, n_out=200):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]

    if len(x) < 8:
        return None, None, None

    xs = np.linspace(x.min(), x.max(), n_out)
    boot = np.empty((n_boot, n_out))

    for b in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        bx, by = loess_fit(x[idx], y[idx], frac=frac)

        if bx is None:
            boot[b, :] = np.nan
        else:
            boot[b, :] = np.interp(xs, bx, by)

    lo = np.nanpercentile(boot, 2.5, axis=0)
    hi = np.nanpercentile(boot, 97.5, axis=0)

    return xs, lo, hi



In [ ]:
# ============================================================
# Figure 1 — 12 biomarqueurs par semestre
# Version finale pour manuscrit
# - pas de titre dans l'image
# - police agrandie
# - figure plus lisible
# - rupture d'axe pour biomarqueurs toujours positifs
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})


# ============================================================
# Paramètres graphiques
# ============================================================

PANEL_TITLE_SIZE = 18
AXIS_LABEL_SIZE = 16
TICK_LABEL_SIZE = 14
BRACKET_SIZE = 13
FOOTNOTE_SIZE = 11

AXIS_COLOR = "#333333"
AXIS_LINEWIDTH = 0.9


# ============================================================
# Biomarqueurs pour lesquels ajouter une référence à 0
# avec cassure d’axe
#
# Les noms doivent correspondre exactement à BIOMARKERS.
# ============================================================

ZERO_BREAK_BIOMARKERS = {
    "MCP-1 pg/mL",
    "Eotaxin pg/mL",
    'IL-8 pg/mL',
    "sICAM-1 (pg/mL)",
    "sVCAM-1 (pg/mL)",
    'VEGF-A pg/mL',
    "MMP-9 (ng/mL)"
}


# ============================================================
# Positions sous chaque panel
# (coordonnées relatives au panel)
# ============================================================

# Ligne supplémentaire correspondant à 0
ZERO_AXIS_Y = -0.085

# Position des dates
X_LABEL_Y = -0.115

# Brackets
BRACKET_TIP_Y = -0.340
BRACKET_Y = -0.375
BRACKET_TEXT_Y = -0.425


# ============================================================
# Fonction : dessiner les axes standards
# (uniquement pour les panels sans cassure)
# ============================================================

def draw_standard_axes(ax):
    """
    Dessine manuellement :
    - l’axe X au bas de la zone graphique ;
    - l’axe Y à gauche.
    """

    # Axe X
    ax.plot(
        [0, 1],
        [0, 0],
        transform=ax.transAxes,
        color=AXIS_COLOR,
        linewidth=AXIS_LINEWIDTH,
        clip_on=False,
        zorder=15
    )

    # Axe Y
    ax.plot(
        [0, 0],
        [0, 1],
        transform=ax.transAxes,
        color=AXIS_COLOR,
        linewidth=AXIS_LINEWIDTH,
        clip_on=False,
        zorder=15
    )


# ============================================================
# Fonction : ajouter l’axe à 0 + cassure
# (sans la ligne horizontale du dessus)
# ============================================================

def add_zero_break(ax, fontsize=TICK_LABEL_SIZE):
    """
    Ajoute sous le panel :
    - une ligne horizontale correspondant à 0 ;
    - un label 0 ;
    - une continuation verticale de l’axe Y ;
    - deux petits traits obliques rapprochés.

    L’échelle des données n’est pas modifiée.
    """

    # Position de la cassure
    break_center_y = -0.043

    # Géométrie des deux traits
    slash_spacing = 0.009
    slash_width = 0.014
    slash_height = 0.014

    slash_centers = [
        break_center_y - slash_spacing / 2,
        break_center_y + slash_spacing / 2
    ]

    # --------------------------------------------------------
    # Ligne horizontale supplémentaire correspondant à 0
    # --------------------------------------------------------

    ax.plot(
        [0, 1],
        [ZERO_AXIS_Y, ZERO_AXIS_Y],
        transform=ax.transAxes,
        color=AXIS_COLOR,
        linewidth=AXIS_LINEWIDTH,
        clip_on=False,
        zorder=20
    )

    # --------------------------------------------------------
    # Label 0
    # --------------------------------------------------------

    ax.text(
        -0.025,
        ZERO_AXIS_Y,
        "0",
        transform=ax.transAxes,
        ha="right",
        va="center",
        fontsize=fontsize,
        color=AXIS_COLOR,
        clip_on=False,
        zorder=25
    )

    # --------------------------------------------------------
    # Partie inférieure de l’axe Y
    # --------------------------------------------------------

    lower_vertical_end = (
        slash_centers[0]
        - slash_height / 2
        - 0.003
    )

    ax.plot(
        [0, 0],
        [ZERO_AXIS_Y, lower_vertical_end],
        transform=ax.transAxes,
        color=AXIS_COLOR,
        linewidth=AXIS_LINEWIDTH,
        clip_on=False,
        zorder=20
    )

    # --------------------------------------------------------
    # Partie supérieure de l’axe Y
    # --------------------------------------------------------

    upper_vertical_start = (
        slash_centers[1]
        + slash_height / 2
        + 0.003
    )

    ax.plot(
        [0, 0],
        [upper_vertical_start, 1],
        transform=ax.transAxes,
        color=AXIS_COLOR,
        linewidth=AXIS_LINEWIDTH,
        clip_on=False,
        zorder=20
    )

    # --------------------------------------------------------
    # Deux traits obliques rapprochés
    # --------------------------------------------------------

    for slash_center in slash_centers:

        ax.plot(
            [-slash_width, slash_width],
            [
                slash_center - slash_height / 2,
                slash_center + slash_height / 2
            ],
            transform=ax.transAxes,
            color=AXIS_COLOR,
            linewidth=1.2,
            solid_capstyle="round",
            clip_on=False,
            zorder=30
        )


# ============================================================
# Fonction : ajouter les dates de l’axe X
# ============================================================

def add_x_labels(ax):
    """
    Ajoute manuellement les semestres sous l’axe X.
    """

    transform = ax.get_xaxis_transform()

    for semester in ORDER:

        ax.text(
            xpos[semester],
            X_LABEL_Y,
            semester,
            transform=transform,
            rotation=45,
            rotation_mode="anchor",
            ha="right",
            va="top",
            fontsize=TICK_LABEL_SIZE,
            fontweight="medium",
            color="#222222",
            clip_on=False,
            zorder=20
        )


# ============================================================
# Fonction : ajouter un bracket
# ============================================================

def add_bracket(ax, x0, x1, text):
    """
    Ajoute un bracket sous l’axe X.
    """

    transform = ax.get_xaxis_transform()

    ax.plot(
        [x0, x0, x1, x1],
        [
            BRACKET_TIP_Y,
            BRACKET_Y,
            BRACKET_Y,
            BRACKET_TIP_Y
        ],
        transform=transform,
        color="#333333",
        linewidth=1.0,
        clip_on=False,
        zorder=20
    )

    ax.text(
        (x0 + x1) / 2,
        BRACKET_TEXT_Y,
        text,
        transform=transform,
        ha="center",
        va="top",
        fontsize=BRACKET_SIZE,
        fontweight="semibold",
        color="#333333",
        clip_on=False,
        zorder=20
    )


# ============================================================
# Création de la figure
# ============================================================

fig, axes = plt.subplots(
    nrows=3,
    ncols=4,
    figsize=(25, 15.5)
)

axes = axes.flatten()

tick_positions = [xpos[semester] for semester in ORDER]


# ============================================================
# Boucle sur les biomarqueurs
# ============================================================

for ax, biomarker in zip(axes, BIOMARKERS):

    yname = biomarker + "_log"
    color = COLOR_MAP[biomarker]
    label = LABELS.get(biomarker, biomarker)

    # --------------------------------------------------------
    # Points individuels par semestre
    # --------------------------------------------------------

    for semester in ORDER:

        vals = (
            df_fig.loc[
                df_fig["semester"] == semester,
                yname
            ]
            .dropna()
            .to_numpy()
        )

        ax.scatter(
            np.full(len(vals), xpos[semester]),
            vals,
            s=6,
            color=color,
            alpha=0.14,
            edgecolors="none",
            zorder=2,
            rasterized=True
        )

    # --------------------------------------------------------
    # Données pour le LOESS
    # --------------------------------------------------------

    sub = (
        df_fig[[yname, "semester"]]
        .dropna()
        .copy()
    )

    sub["xnum"] = sub["semester"].map(xpos)
    sub = sub.dropna(subset=["xnum"])

    xnum = sub["xnum"].to_numpy()
    ynum = sub[yname].to_numpy()

    # --------------------------------------------------------
    # LOESS et intervalle de confiance bootstrap
    # --------------------------------------------------------

    if len(sub) > 0:

        lx, ly = loess_fit(
            xnum,
            ynum,
            frac=0.7
        )

        if lx is not None:

            cx, clo, chi = loess_ci(
                xnum,
                ynum,
                frac=0.7,
                n_boot=200
            )

            if cx is not None:

                ax.fill_between(
                    cx,
                    clo,
                    chi,
                    color=color,
                    alpha=0.16,
                    linewidth=0,
                    zorder=3
                )

            ax.plot(
                lx,
                ly,
                color=color,
                linewidth=2.4,
                solid_capstyle="round",
                zorder=4
            )

    # --------------------------------------------------------
    # Médiane observée par semestre
    # --------------------------------------------------------

    medians = (
        sub.groupby(
            "semester",
            observed=False
        )[yname]
        .median()
        .reindex(ORDER)
    )

    median_x = [
        xpos[semester]
        for semester in ORDER
        if pd.notna(medians.loc[semester])
    ]

    median_y = [
        medians.loc[semester]
        for semester in ORDER
        if pd.notna(medians.loc[semester])
    ]

    ax.plot(
        median_x,
        median_y,
        linestyle="--",
        color="#222222",
        linewidth=1.8,
        dash_capstyle="round",
        zorder=5
    )

    # --------------------------------------------------------
    # Titre du panel
    # --------------------------------------------------------

    ax.set_title(
        label,
        fontsize=PANEL_TITLE_SIZE,
        fontweight="bold",
        loc="left",
        pad=10,
        bbox=dict(
            boxstyle="round,pad=0.20",
            facecolor="#eeeeee",
            edgecolor="none"
        )
    )

    # --------------------------------------------------------
    # Axe X : labels standards masqués
    # --------------------------------------------------------

    ax.set_xticks(tick_positions)
    ax.set_xticklabels([])

    ax.tick_params(
        axis="x",
        which="major",
        length=0
    )

    # --------------------------------------------------------
    # Axe Y
    # --------------------------------------------------------

    ax.set_ylabel("")

    ax.tick_params(
        axis="y",
        which="major",
        labelsize=TICK_LABEL_SIZE,
        length=0,
        pad=6
    )

    # --------------------------------------------------------
    # Limites
    # --------------------------------------------------------

    ax.set_xlim(
        min(tick_positions) - 0.5,
        max(tick_positions) + 0.5
    )

    ax.margins(y=0.06)

    # --------------------------------------------------------
    # Fond et grille
    # --------------------------------------------------------

    ax.set_axisbelow(True)

    ax.grid(
        axis="y",
        color="#d9d9d9",
        linewidth=0.8,
        alpha=0.65
    )

    ax.grid(
        axis="x",
        visible=False
    )

    ax.set_facecolor("white")

    # Masquer les bordures par défaut
    for spine in ax.spines.values():
        spine.set_visible(False)

    # --------------------------------------------------------
    # Vérifier si le biomarqueur est toujours positif
    # --------------------------------------------------------

    panel_is_positive = (
        len(ynum) > 0
        and np.isfinite(ynum).any()
        and np.nanmin(ynum) > 0
    )

    # --------------------------------------------------------
    # Dessiner les axes
    # --------------------------------------------------------

    if (
        biomarker in ZERO_BREAK_BIOMARKERS
        and panel_is_positive
    ):

        add_zero_break(
            ax=ax,
            fontsize=TICK_LABEL_SIZE
        )

    else:

        draw_standard_axes(ax)

    # --------------------------------------------------------
    # Ajouter les dates
    # --------------------------------------------------------

    add_x_labels(ax)

    # --------------------------------------------------------
    # Brackets sur tous les panels
    # --------------------------------------------------------

    add_bracket(
        ax=ax,
        x0=xpos[ORDER[0]] - 0.10,
        x1=xpos[ORDER[1]] + 0.10,
        text="Pre-pandemic"
    )

    add_bracket(
        ax=ax,
        x0=xpos[ORDER[2]] - 0.10,
        x1=xpos[ORDER[-1]] + 0.10,
        text="Post-pandemic"
    )


# ============================================================
# Masquer d’éventuels panels inutilisés
# ============================================================

for ax in axes[len(BIOMARKERS):]:
    ax.set_visible(False)


# ============================================================
# Label Y global
# ============================================================

fig.supylabel(
    "Log biomarker value",
    x=0.012,
    fontsize=AXIS_LABEL_SIZE
)


# ============================================================
# Note de bas de figure
# (pas de titre dans l'image)
# ============================================================



# ============================================================
# Espacement
# ============================================================

plt.subplots_adjust(
    left=0.048,
    right=0.997,
    top=0.980,
    bottom=0.145,
    wspace=0.06,
    hspace=0.68
)

# ============================================================
# Export
# ============================================================

fig_dir = FIGURES_DIR
fig_dir.mkdir(
    parents=True,
    exist_ok=True
)

base_name = (
    "Figure1_inflammatory_biomarker_"
    "trajectories_by_semester_final_manuscript"
)

png_path = fig_dir / f"{base_name}_600dpi.png"
pdf_path = fig_dir / f"{base_name}.pdf"
svg_path = fig_dir / f"{base_name}.svg"

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

fig.savefig(
    svg_path,
    bbox_inches="tight",
    pad_inches=0.08,
    facecolor="white"
)

plt.show()

print("Figure saved to:")
print(png_path)
print(pdf_path)
print(svg_path)

## 5. Figure 2 — Main adjusted pre/post-pandemic differences

In [ ]:
from matplotlib.lines import Line2D

covariates_model = [
    "period", "age", "sex_en", "education_3cat_model", "income_3cat_model",
    "bmi_nursemeasured", "smoking_status_model", "season"
]

formula = (
    "log_biomarker ~ C(period, Treatment(reference='pre_pandemic'))"
    " + age + C(sex_en) + C(education_3cat_model) + C(income_3cat_model)"
    " + bmi_nursemeasured + C(smoking_status_model) + C(season)"
)
coef_name = "C(period, Treatment(reference='pre_pandemic'))[T.post_pandemic]"

main_results = []
for biomarker in biomarkers:
    tmp = df[[biomarker] + covariates_model].copy()
    tmp[biomarker] = pd.to_numeric(tmp[biomarker], errors="coerce")
    tmp = tmp[tmp[biomarker] > 0].dropna(subset=[biomarker] + covariates_model).copy()
    tmp["period"] = pd.Categorical(tmp["period"], ["pre_pandemic", "post_pandemic"], ordered=True)
    tmp["log_biomarker"] = np.log(tmp[biomarker])

    fit = smf.ols(formula, data=tmp).fit()
    beta = fit.params[coef_name]
    lo, hi = fit.conf_int().loc[coef_name]
    main_results.append({
        "biomarker": biomarker,
        "n": int(fit.nobs),
        "percent_difference": 100 * (np.exp(beta) - 1),
        "ci_low": 100 * (np.exp(lo) - 1),
        "ci_high": 100 * (np.exp(hi) - 1),
        "p_value": fit.pvalues[coef_name],
    })

figure2_df = pd.DataFrame(main_results)
figure2_df["p_fdr"] = multipletests(figure2_df["p_value"], method="fdr_bh")[1]
figure2_df["fdr_group"] = np.where(
    figure2_df["p_fdr"] < 0.05,
    "FDR-adjusted P < 0.05",
    "FDR-adjusted P ≥ 0.05",
)

figure2_df.to_excel(RESULTS_DIR / "Figure2_main_adjusted_results.xlsx", index=False)
display(figure2_df)

# Forest plot — same statistical results, neutral IJE wording rather than significant/non-significant.
plot_order = biomarkers
plot_df = figure2_df.set_index("biomarker").reindex(plot_order).reset_index()
plot_df["y"] = np.arange(len(plot_df))[::-1]

color_map = {
    "FDR-adjusted P < 0.05": "#C44E52",
    "FDR-adjusted P ≥ 0.05": "#4C72B0",
}

plt.rcParams.update({"font.family": "Arial", "pdf.fonttype": 42, "ps.fonttype": 42})
fig, ax = plt.subplots(figsize=(8.5, 6.5))
for _, row in plot_df.iterrows():
    ax.errorbar(
        row["percent_difference"], row["y"],
        xerr=[[row["percent_difference"] - row["ci_low"]], [row["ci_high"] - row["percent_difference"]]],
        fmt="o", capsize=3, linewidth=1.2, markersize=6,
        color=color_map[row["fdr_group"]]
    )

ax.axvline(0, linestyle="--", linewidth=1, color="black")
ax.set_yticks(plot_df["y"])
ax.set_yticklabels(plot_df["biomarker"], fontsize=10)
ax.set_xlabel("Adjusted percentage difference post- vs pre-pandemic (%)", fontsize=11)
ax.set_ylabel("")
ax.text(0.25, -0.13, "← Lower post-pandemic levels", transform=ax.transAxes, ha="center", va="top", fontsize=10)
ax.text(0.75, -0.13, "Higher post-pandemic levels →", transform=ax.transAxes, ha="center", va="top", fontsize=10)

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[k], markersize=7, label=k)
    for k in ["FDR-adjusted P < 0.05", "FDR-adjusted P ≥ 0.05"]
]
ax.legend(handles=legend_elements, title="FDR-adjusted P value", loc="upper right", frameon=True, fontsize=10, title_fontsize=10)
ax.grid(axis="x", alpha=0.25)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
fig.savefig(FIGURES_DIR / "Figure2_adjusted_prepost_forest_plot_IJE.png", dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(FIGURES_DIR / "Figure2_adjusted_prepost_forest_plot_IJE.pdf", bbox_inches="tight", facecolor="white")
plt.show()

## 6. Table S2 — Sequential regression models

In [ ]:
# ============================================================
# Table 3 — Sequential regression models
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# 1. Recode variables if needed
# ------------------------------------------------------------

df["age"] = df["age.x"]

df = df.rename(columns={
    "Moderate PA, standardized data, PAFQ (min/day)": "PAFQ_MPA_raw",
    "Vigorous PA, standardized data, PAFQ (min/day)": "PAFQ_VPA_raw",
    "Sedentary, standardized data, PAFQ (min/day)": "PAFQ_SE_raw"
})

df["education_3cat"] = df["m_education_coded"].replace({
    "Primary": "Primary",
    "Secondary": "Secondary",
    "Tertiary": "Tertiary",
    "Autre": np.nan
})

df["income_3cat"] = df["m_income_grp"].replace({
    "Faible": "Low",
    "Modéré faible": "Low",
    "Modéré élevé": "Middle",
    "Elevé": "High",
    "Inconnu": np.nan,
    "Refus": np.nan
})

df["education_3cat_model"] = df["education_3cat"].fillna("Missing")
df["income_3cat_model"] = df["income_3cat"].fillna("Missing")
df["smoking_status_model"] = df["smoking_status"].fillna("Missing")

# ============================================================
# Create physical activity variable
# ============================================================

df["PAFQ_MPA_raw"] = pd.to_numeric(df["PAFQ_MPA_raw"], errors="coerce")
df["PAFQ_VPA_raw"] = pd.to_numeric(df["PAFQ_VPA_raw"], errors="coerce")

df["physical_activity_min_day"] = df["PAFQ_MPA_raw"] + df["PAFQ_VPA_raw"]

lifestyle_physical_activity_sleep_vars = [
    "physical_activity_min_day",
    "PAFQ_SE_raw",
    "somttot",
    "sumalco"
]

# ------------------------------------------------------------
# 2. Define model specifications
# ------------------------------------------------------------

model_specs = {
    "Model 0: crude": [
        "period"
    ],
    
    "Model 1: age sex season": [
        "period",
        "age",
        "sex_en",
        "season"
    ],
    
    "Model 2: main adjusted": [
        "period",
        "age",
        "sex_en",
        "education_3cat_model",
        "income_3cat_model",
        "bmi_nursemeasured",
        "smoking_status_model",
        "season"
    ],
    
    "Model 3: + cardiometabolic": [
        "period",
        "age",
        "sex_en",
        "education_3cat_model",
        "income_3cat_model",
        "bmi_nursemeasured",
        "smoking_status_model",
        "season",
        "self_reported_and_detected_diabetes",
        "self_reported_and_detected_HTA",
        "self_reported_and_detected_high_cholesterol"
    ],

    "Model 4: + lifestyle physical activity sleep": [
        "period",
        "age",
        "sex_en",
        "education_3cat_model",
        "income_3cat_model",
        "bmi_nursemeasured",
        "smoking_status_model",
        "season",
        "self_reported_and_detected_diabetes",
        "self_reported_and_detected_HTA",
        "self_reported_and_detected_high_cholesterol",
        "physical_activity_min_day",
        "PAFQ_SE_raw",
        "somttot",
        "sumalco"
    ]
}

categorical_covariates = [
    "period",
    "sex_en",
    "education_3cat_model",
    "income_3cat_model",
    "smoking_status_model",
    "season",
    "self_reported_and_detected_diabetes",
    "self_reported_and_detected_HTA",
    "self_reported_and_detected_high_cholesterol",
]

# ------------------------------------------------------------
# 3. Function to run one model specification on all biomarkers
# ------------------------------------------------------------

def run_biomarker_model(df, biomarkers, covariates, model_name):
    
    results = []
    
    for biomarker in biomarkers:
        
        needed_cols = [biomarker] + covariates
        tmp = df[needed_cols].copy()
        
        tmp[biomarker] = pd.to_numeric(tmp[biomarker], errors="coerce")
        tmp = tmp.dropna()
        tmp = tmp[tmp[biomarker] > 0]
        
        tmp["period"] = pd.Categorical(
            tmp["period"],
            categories=["pre_pandemic", "post_pandemic"],
            ordered=True
        )
        
        tmp["log_biomarker"] = np.log(tmp[biomarker])
        
        formula_terms = []
        
        for var in covariates:
            if var == "period":
                formula_terms.append("C(period, Treatment(reference='pre_pandemic'))")
            elif var in categorical_covariates:
                formula_terms.append(f"C({var})")
            else:
                formula_terms.append(var)
        
        formula = "log_biomarker ~ " + " + ".join(formula_terms)
        
        model = smf.ols(formula, data=tmp).fit()
        
        coef_name = "C(period, Treatment(reference='pre_pandemic'))[T.post_pandemic]"
        
        beta = model.params[coef_name]
        ci_low, ci_high = model.conf_int().loc[coef_name]
        pval = model.pvalues[coef_name]
        
        pct_change = (np.exp(beta) - 1) * 100
        pct_low = (np.exp(ci_low) - 1) * 100
        pct_high = (np.exp(ci_high) - 1) * 100
        
        results.append({
            "model": model_name,
            "biomarker": biomarker,
            "n": len(tmp),
            "percent_difference": pct_change,
            "ci_low": pct_low,
            "ci_high": pct_high,
            "p_value": pval,
            "formula": formula
        })
    
    return pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Run all models
# ------------------------------------------------------------

all_results = []

for model_name, covariates in model_specs.items():
    res = run_biomarker_model(
        df=df,
        biomarkers=biomarkers,
        covariates=covariates,
        model_name=model_name
    )
    all_results.append(res)

table3_full = pd.concat(all_results, ignore_index=True)

# ------------------------------------------------------------
# 5. Format results
# ------------------------------------------------------------

table3_full["effect_95CI"] = table3_full.apply(
    lambda row: f"{row['percent_difference']:.1f}% "
                f"[{row['ci_low']:.1f}, {row['ci_high']:.1f}]",
    axis=1
)

table3_full["p_value_formatted"] = table3_full["p_value"].apply(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
)

# ============================================================
# Add FDR correction within each model
# ============================================================

from statsmodels.stats.multitest import multipletests

table3_full = table3_full.copy()

table3_full["p_fdr"] = np.nan

for model_name in table3_full["model"].unique():
    
    idx = table3_full["model"] == model_name
    
    table3_full.loc[idx, "p_fdr"] = multipletests(
        table3_full.loc[idx, "p_value"],
        method="fdr_bh"
    )[1]


def format_p(p):
    if pd.isna(p):
        return "—"
    elif p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"


table3_full["p_value_formatted"] = table3_full["p_value"].apply(format_p)
table3_full["p_fdr_formatted"] = table3_full["p_fdr"].apply(format_p)

table3_full["effect_95CI"] = table3_full.apply(
    lambda row: f"{row['percent_difference']:.1f}% "
                f"[{row['ci_low']:.1f}, {row['ci_high']:.1f}]",
    axis=1
)

table3_full["effect_95CI_pFDR"] = table3_full.apply(
    lambda row: f"{row['effect_95CI']}, pFDR={row['p_fdr_formatted']}",
    axis=1
)

table3_long = table3_full[[
    "model",
    "biomarker",
    "n",
    "effect_95CI",
    "p_value_formatted"
]]

table3_long

In [ ]:
# Wide manuscript-ready version, with FDR-adjusted P values within each model.
model_order = list(model_specs.keys())
wide = table3_full.copy()
wide["display"] = wide.apply(
    lambda r: f"{r['percent_difference']:.1f}% [{r['ci_low']:.1f}, {r['ci_high']:.1f}], P=" +
              ("<0.001" if r["p_fdr"] < 0.001 else f"{r['p_fdr']:.3f}"),
    axis=1
)
table_s2 = wide.pivot(index="biomarker", columns="model", values="display").reindex(index=biomarkers, columns=model_order).reset_index()
table_s2.columns.name = None
table_s2.to_excel(RESULTS_DIR / "Table_S2_sequential_models.xlsx", index=False)
display(table_s2)

## 7. Table S3 — Year-specific adjusted differences

In [ ]:
# ============================================================
# Sensitivity analysis:
# Adjusted biomarker differences by sampling year
# 2023 vs 2019 and 2024 vs 2019
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests


# ------------------------------------------------------------
# 1. Main Model 2 adjustment set
# ------------------------------------------------------------

main_covariates = [
    "age",
    "sex_en",
    "education_3cat_model",
    "income_3cat_model",
    "bmi_nursemeasured",
    "smoking_status_model",
    "season"
]

categorical_covariates_year = {
    "sex_en",
    "education_3cat_model",
    "income_3cat_model",
    "smoking_status_model",
    "season"
}

biomarkers_year_order = [
    "CRP mg/l",
    "MCP-1 pg/mL",
    "TNF-α pg/mL",
    "IFNγ pg/mL",
    "Eotaxin pg/mL",
    "IL-6 pg/mL",
    "IL-8 pg/mL",
    "IL-10 pg/mL",
    "sICAM-1 (pg/mL)",
    "sVCAM-1 (pg/mL)",
    "VEGF-A pg/mL",
    "MMP-9 (ng/mL)"
]

# Keep only biomarkers present in the dataframe
biomarkers_year_order = [
    biomarker
    for biomarker in biomarkers_year_order
    if biomarker in df.columns
]


# ------------------------------------------------------------
# 2. Run one categorical-year model for each biomarker
# ------------------------------------------------------------

year_results = []

for biomarker in biomarkers_year_order:

    needed_columns = [
        biomarker,
        "year"
    ] + main_covariates

    tmp = df[needed_columns].copy()

    # Ensure numeric biomarker values
    tmp[biomarker] = pd.to_numeric(
        tmp[biomarker],
        errors="coerce"
    )

    # Ensure numeric year values
    tmp["year"] = pd.to_numeric(
        tmp["year"],
        errors="coerce"
    )

    # Retain only the three study years
    tmp = tmp[
        tmp["year"].isin([2019, 2023, 2024])
    ].copy()

    # Complete cases for the current biomarker/model
    tmp = tmp.dropna(subset=needed_columns)

    # Log transformation requires positive concentrations
    tmp = tmp[
        tmp[biomarker] > 0
    ].copy()

    # Define 2019 as the reference year
    tmp["year"] = (
        tmp["year"]
        .astype(int)
        .astype(str)
    )

    tmp["year"] = pd.Categorical(
        tmp["year"],
        categories=["2019", "2023", "2024"],
        ordered=True
    )

    tmp["log_biomarker"] = np.log(
        tmp[biomarker]
    )

    # Model 2 formula
    formula_terms = [
        "C(year, Treatment(reference='2019'))"
    ]

    for variable in main_covariates:

        if variable in categorical_covariates_year:
            formula_terms.append(
                f"C({variable})"
            )
        else:
            formula_terms.append(variable)

    formula = (
        "log_biomarker ~ "
        + " + ".join(formula_terms)
    )

    model = smf.ols(
        formula,
        data=tmp
    ).fit()

    # Coefficient names generated by statsmodels
    year_coefficients = {
        "2023 vs 2019":
            "C(year, Treatment(reference='2019'))[T.2023]",
        "2024 vs 2019":
            "C(year, Treatment(reference='2019'))[T.2024]"
    }

    for contrast, coefficient_name in year_coefficients.items():

        beta = model.params[
            coefficient_name
        ]

        ci_low_log, ci_high_log = (
            model.conf_int()
            .loc[coefficient_name]
        )

        p_value = model.pvalues[
            coefficient_name
        ]

        year_results.append({
            "biomarker": biomarker,
            "contrast": contrast,
            "n": int(model.nobs),
            "beta_log": beta,
            "percent_difference":
                100 * (np.exp(beta) - 1),
            "ci_low":
                100 * (np.exp(ci_low_log) - 1),
            "ci_high":
                100 * (np.exp(ci_high_log) - 1),
            "p_value": p_value
        })


year_results = pd.DataFrame(
    year_results
)


# ------------------------------------------------------------
# 3. FDR correction
# Separately across the 12 biomarkers for each comparison
# ------------------------------------------------------------

year_results["p_fdr"] = np.nan

for contrast in [
    "2023 vs 2019",
    "2024 vs 2019"
]:

    mask = (
        year_results["contrast"] == contrast
    )

    year_results.loc[
        mask,
        "p_fdr"
    ] = multipletests(
        year_results.loc[
            mask,
            "p_value"
        ],
        method="fdr_bh"
    )[1]


# ------------------------------------------------------------
# 4. Format results
# ------------------------------------------------------------

def format_p_value(p):

    if pd.isna(p):
        return "—"

    if p < 0.001:
        return "<0.001"

    return f"{p:.3f}"


year_results["p_fdr_formatted"] = (
    year_results["p_fdr"]
    .apply(format_p_value)
)

year_results["result"] = (
    year_results.apply(
        lambda row:
            f"{row['percent_difference']:+.1f}% "
            f"[{row['ci_low']:+.1f}, "
            f"{row['ci_high']:+.1f}], "
            f"pFDR={row['p_fdr_formatted']}",
        axis=1
    )
)


# ------------------------------------------------------------
# 5. Wide supplementary table
# ------------------------------------------------------------

table_year_sensitivity = (
    year_results
    .pivot(
        index="biomarker",
        columns="contrast",
        values="result"
    )
    .reindex(columns=[
        "2023 vs 2019",
        "2024 vs 2019"
    ])
    .reset_index()
)

table_year_sensitivity.columns.name = None

# Restore biomarker order
table_year_sensitivity["biomarker"] = pd.Categorical(
    table_year_sensitivity["biomarker"],
    categories=biomarkers_year_order,
    ordered=True
)

table_year_sensitivity = (
    table_year_sensitivity
    .sort_values("biomarker")
    .reset_index(drop=True)
)

table_year_sensitivity["biomarker"] = (
    table_year_sensitivity["biomarker"]
    .astype(str)
)

display(table_year_sensitivity)


# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

output_directory = RESULTS_DIR

output_directory.mkdir(
    parents=True,
    exist_ok=True
)

table_year_sensitivity.to_excel(
    output_directory
    / "Table_S_year_specific_adjusted_biomarker_differences.xlsx",
    index=False
)

year_results.to_excel(
    output_directory
    / "Table_S_year_specific_adjusted_biomarker_differences_long.xlsx",
    index=False
)

## 8. Table S4 — Principal component analysis

In [ ]:
# ============================================================
# PCA of the 12 inflammatory biomarkers
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# 1. Biomarkers
# ------------------------------------------------------------

pca_biomarkers = [
    "CRP mg/l",
    "MCP-1 pg/mL",
    "TNF-α pg/mL",
    "IFNγ pg/mL",
    "Eotaxin pg/mL",
    "IL-6 pg/mL",
    "IL-8 pg/mL",
    "IL-10 pg/mL",
    "sICAM-1 (pg/mL)",
    "sVCAM-1 (pg/mL)",
    "VEGF-A pg/mL",
    "MMP-9 (ng/mL)"
]

pca_biomarkers = [
    marker
    for marker in pca_biomarkers
    if marker in df.columns
]

# ------------------------------------------------------------
# 2. Prepare biomarker matrix
# ------------------------------------------------------------

pca_data = df.copy()

for marker in pca_biomarkers:

    pca_data[marker] = pd.to_numeric(
        pca_data[marker],
        errors="coerce"
    )

    # Log-transform requires positive concentrations
    pca_data.loc[
        pca_data[marker] <= 0,
        marker
    ] = np.nan

# PCA needs the same participants for all biomarkers.
# Complete-case PCA is preferable if missingness is limited.
pca_complete = pca_data.dropna(
    subset=pca_biomarkers
).copy()

print(
    f"PCA complete cases: {len(pca_complete)} "
    f"of {len(pca_data)} participants "
    f"({100 * len(pca_complete) / len(pca_data):.1f}%)"
)

# Log-transform
log_biomarkers = np.log(
    pca_complete[pca_biomarkers]
)

# Standardize each biomarker
scaler = StandardScaler()

X_standardized = scaler.fit_transform(
    log_biomarkers
)

# ------------------------------------------------------------
# 3. Fit PCA
# ------------------------------------------------------------

pca = PCA()

pca_scores = pca.fit_transform(
    X_standardized
)

pc_names = [
    f"PC{i + 1}"
    for i in range(len(pca_biomarkers))
]

# Eigenvector coefficients
component_weights = pd.DataFrame(
    pca.components_.T,
    index=pca_biomarkers,
    columns=pc_names
)

# ------------------------------------------------------------
# 4. Orient PC1 for easier interpretation
# PCA signs are arbitrary.
# Make positive PC1 correspond, where possible, to higher
# classical inflammatory markers.
# ------------------------------------------------------------

classical_markers = [
    marker for marker in [
        "CRP mg/l",
        "TNF-α pg/mL",
        "IL-6 pg/mL",
        "IL-8 pg/mL"
    ]
    if marker in component_weights.index
]

if (
    component_weights.loc[
        classical_markers,
        "PC1"
    ].mean() < 0
):
    pca_scores[:, 0] *= -1
    pca.components_[0, :] *= -1

    component_weights = pd.DataFrame(
        pca.components_.T,
        index=pca_biomarkers,
        columns=pc_names
    )

# ------------------------------------------------------------
# 5. Explained variance
# ------------------------------------------------------------

variance_table = pd.DataFrame({
    "component": pc_names,
    "eigenvalue": pca.explained_variance_,
    "explained_variance_percent":
        pca.explained_variance_ratio_ * 100,
    "cumulative_variance_percent":
        np.cumsum(
            pca.explained_variance_ratio_
        ) * 100
})

display(variance_table)

# ------------------------------------------------------------
# 6. PCA loadings
#
# For standardized variables, these approximate correlations
# between each biomarker and each principal component.
# ------------------------------------------------------------

pca_loadings = pd.DataFrame(
    pca.components_.T
    * np.sqrt(pca.explained_variance_),
    index=pca_biomarkers,
    columns=pc_names
)

display(
    pca_loadings[
        ["PC1", "PC2", "PC3"]
    ].round(3)
)

# ------------------------------------------------------------
# 7. Scree plot
# ------------------------------------------------------------

plt.figure(figsize=(7, 4))

plt.plot(
    range(1, len(pca_biomarkers) + 1),
    variance_table["explained_variance_percent"],
    marker="o"
)

plt.xlabel("Principal component")
plt.ylabel("Explained variance (%)")
plt.title("Scree plot of inflammatory biomarkers")
plt.xticks(
    range(1, len(pca_biomarkers) + 1)
)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 8. Attach standardized PC scores to participants
# ------------------------------------------------------------

pca_scored = pca_complete.copy()

# Store at least the first three components.
# Standardize each component score so regression coefficients
# are expressed in SD units.
for component_number in range(3):

    score = pca_scores[:, component_number]

    score_standardized = (
        score - score.mean()
    ) / score.std(ddof=1)

    pca_scored[
        f"PC{component_number + 1}_z"
    ] = score_standardized

In [ ]:
# ============================================================
# Adjusted pre/post-pandemic difference in PCA-derived PC1
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Ensure year is numeric
pca_scored["year"] = pd.to_numeric(
    pca_scored["year"],
    errors="coerce"
)

# 0 = 2019, 1 = 2023-2024
pca_scored["post_pandemic"] = np.where(
    pca_scored["year"] == 2019,
    0,
    1
)

needed_columns = [
    "PC1_z",
    "post_pandemic",
    "age",
    "sex_en",
    "education_3cat_model",
    "income_3cat_model",
    "bmi_nursemeasured",
    "smoking_status_model",
    "season"
]

tmp_pc1 = (
    pca_scored[needed_columns]
    .dropna()
    .copy()
)

formula_pc1 = """
PC1_z ~ post_pandemic
        + age
        + C(sex_en)
        + C(education_3cat_model)
        + C(income_3cat_model)
        + bmi_nursemeasured
        + C(smoking_status_model)
        + C(season)
"""

model_pc1 = smf.ols(
    formula_pc1,
    data=tmp_pc1
).fit()

beta = model_pc1.params["post_pandemic"]
ci_low, ci_high = (
    model_pc1.conf_int()
    .loc["post_pandemic"]
)
p_value = model_pc1.pvalues[
    "post_pandemic"
]

pc1_result = pd.DataFrame({
    "Outcome": ["PC1 score"],
    "n": [int(model_pc1.nobs)],
    "Adjusted difference, SD": [beta],
    "95% CI low": [ci_low],
    "95% CI high": [ci_high],
    "p-value": [p_value]
})

display(pc1_result)

print(
    f"Adjusted post- vs pre-pandemic difference in PC1: "
    f"{beta:+.2f} SD "
    f"[95% CI {ci_low:+.2f} to {ci_high:+.2f}], "
    f"p={p_value:.3f}"
)

In [ ]:
# ============================================================
# Adjusted year-specific differences in PC1
# ============================================================

pca_scored["year_str"] = (
    pd.to_numeric(
        pca_scored["year"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
)

pca_scored = pca_scored[
    pca_scored["year_str"].isin(
        ["2019", "2023", "2024"]
    )
].copy()

pca_scored["year_str"] = pd.Categorical(
    pca_scored["year_str"],
    categories=["2019", "2023", "2024"],
    ordered=True
)

needed_columns_year = [
    "PC1_z",
    "year_str",
    "age",
    "sex_en",
    "education_3cat_model",
    "income_3cat_model",
    "bmi_nursemeasured",
    "smoking_status_model",
    "season"
]

tmp_pc1_year = (
    pca_scored[needed_columns_year]
    .dropna()
    .copy()
)

formula_pc1_year = """
PC1_z ~ C(year_str, Treatment(reference='2019'))
        + age
        + C(sex_en)
        + C(education_3cat_model)
        + C(income_3cat_model)
        + bmi_nursemeasured
        + C(smoking_status_model)
        + C(season)
"""

model_pc1_year = smf.ols(
    formula_pc1_year,
    data=tmp_pc1_year
).fit()

year_results_pc1 = []

for year in ["2023", "2024"]:

    coefficient = (
        "C(year_str, Treatment(reference='2019'))"
        f"[T.{year}]"
    )

    beta = model_pc1_year.params[
        coefficient
    ]

    ci_low, ci_high = (
        model_pc1_year.conf_int()
        .loc[coefficient]
    )

    p_value = model_pc1_year.pvalues[
        coefficient
    ]

    year_results_pc1.append({
        "Comparison": f"{year} vs 2019",
        "n": int(model_pc1_year.nobs),
        "Adjusted difference, SD": beta,
        "95% CI low": ci_low,
        "95% CI high": ci_high,
        "p-value": p_value
    })

year_results_pc1 = pd.DataFrame(
    year_results_pc1
)

display(year_results_pc1)

In [ ]:
with pd.ExcelWriter(RESULTS_DIR / "Table_S4_PCA.xlsx") as writer:
    variance_table.to_excel(writer, sheet_name="Variance", index=False)
    pca_loadings[["PC1", "PC2", "PC3"]].round(3).to_excel(writer, sheet_name="Loadings")
    pc1_result.to_excel(writer, sheet_name="PC1_prepost", index=False)
    year_results_pc1.to_excel(writer, sheet_name="PC1_year_specific", index=False)
print("Saved:", RESULTS_DIR / "Table_S4_PCA.xlsx")

## 9. Tables S5–S6 — Lifestyle analyses and effect modification

In [ ]:
# ============================================================
# Exploratory lifestyle analyses:
# Table S5 = associations of lifestyle categories with biomarkers
# Table S6 = stratified pre/post-pandemic differences by lifestyle categories
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# ------------------------------------------------------------
# 0. Output folder
# ------------------------------------------------------------

OUTPUT_DIR = RESULTS_DIR / "lifestyle_tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Data copy
# ------------------------------------------------------------

df_life = df.copy()

# Post-pandemic indicator
if "post" not in df_life.columns:
    df_life["post"] = np.where(df_life["period"] == "post_pandemic", 1, 0)

# ------------------------------------------------------------
# 2. Biomarker order and display labels
# ------------------------------------------------------------

BIOMARKER_SPECS = [
    {"col": "CRP mg/l",        "label": "CRP"},
    {"col": "MCP-1 pg/mL",     "label": "MCP-1"},
    {"col": "TNF-α pg/mL",     "label": "TNF-α"},
    {"col": "IFNγ pg/mL",      "label": "IFNγ"},
    {"col": "Eotaxin pg/mL",   "label": "eotaxin"},
    {"col": "IL-6 pg/mL",      "label": "IL-6"},
    {"col": "IL-8 pg/mL",      "label": "IL-8"},
    {"col": "IL-10 pg/mL",     "label": "IL-10"},
    {"col": "sICAM-1 (pg/mL)", "label": "sICAM-1"},
    {"col": "sVCAM-1 (pg/mL)", "label": "sVCAM-1"},
    {"col": "VEGF-A pg/mL",    "label": "VEGF-A"},
    {"col": "MMP-9 (ng/mL)",   "label": "MMP-9"},
]

BIOMARKERS = [x["col"] for x in BIOMARKER_SPECS]
BIOMARKER_LABELS = {x["col"]: x["label"] for x in BIOMARKER_SPECS}
BIOMARKER_ORDER = [x["label"] for x in BIOMARKER_SPECS]

# ------------------------------------------------------------
# 3. Column names
# ------------------------------------------------------------

AGE_COL = "age"
SEX_COL = "sex_en"
EDUC_COL = "education_3cat_model"
INCOME_COL = "income_3cat_model"
SMOKING_COL = "smoking_status_model"
SEASON_COL = "season"

# Use measured BMI, not obesity, to match Methods / main model.
# Adapt if your actual measured BMI column has another name.
if "bmi_nursemeasured" in df_life.columns:
    BMI_COL = "bmi_nursemeasured"
elif "bmi" in df_life.columns:
    BMI_COL = "bmi"
else:
    raise ValueError("No measured BMI column found. Check BMI column name.")

# Lifestyle source columns
MPA_COL = "PAFQ_MPA_raw"
VPA_COL = "PAFQ_VPA_raw"
SEDENTARY_COL = "PAFQ_SE_raw"
SLEEP_COL = "somttot"
ALCOHOL_COL = "sumalco"

needed_cols = [
    "post", AGE_COL, SEX_COL, EDUC_COL, INCOME_COL, BMI_COL,
    SMOKING_COL, SEASON_COL, MPA_COL, VPA_COL,
    SEDENTARY_COL, SLEEP_COL, ALCOHOL_COL
] + BIOMARKERS

missing_cols = [c for c in needed_cols if c not in df_life.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Convert numeric variables
for col in [AGE_COL, BMI_COL, MPA_COL, VPA_COL, SEDENTARY_COL, SLEEP_COL, ALCOHOL_COL] + BIOMARKERS:
    df_life[col] = pd.to_numeric(df_life[col], errors="coerce")

# Total physical activity = moderate + vigorous physical activity
df_life["total_physical_activity"] = df_life[MPA_COL] + df_life[VPA_COL]

# ------------------------------------------------------------
# 4. Create lifestyle categories
# ------------------------------------------------------------

def make_tertiles(x, labels):
    """
    Creates tertiles. If qcut fails because of many tied values,
    uses ranked values to preserve three ordered groups.
    """
    x = pd.to_numeric(x, errors="coerce")
    try:
        return pd.qcut(x, q=3, labels=labels)
    except ValueError:
        return pd.qcut(x.rank(method="first"), q=3, labels=labels)

df_life["physical_activity_group"] = make_tertiles(
    df_life["total_physical_activity"],
    ["Low physical activity", "Medium physical activity", "High physical activity"]
)

df_life["sedentary_group"] = make_tertiles(
    df_life[SEDENTARY_COL],
    ["Low sedentary time", "Medium sedentary time", "High sedentary time"]
)

df_life["alcohol_group"] = make_tertiles(
    df_life[ALCOHOL_COL],
    ["Low alcohol intake", "Medium alcohol intake", "High alcohol intake"]
)

df_life["sleep_group"] = pd.cut(
    df_life[SLEEP_COL],
    bins=[-np.inf, 49, 63, np.inf],
    labels=["Short sleep", "Normal sleep", "Long sleep"]
)

LIFESTYLE_SPECS = [
    {
        "var": "physical_activity_group",
        "label": "Physical activity",
        "reference": "Low physical activity",
    },
    {
        "var": "sedentary_group",
        "label": "Sedentary time",
        "reference": "Low sedentary time",
    },
    {
        "var": "sleep_group",
        "label": "Sleep duration",
        "reference": "Normal sleep",
    },
    {
        "var": "alcohol_group",
        "label": "Alcohol intake",
        "reference": "Low alcohol intake",
    },
]

# ------------------------------------------------------------
# 5. Covariates
# ------------------------------------------------------------

COVARIATES_NO_POST = [
    AGE_COL,
    f"C({SEX_COL})",
    f"C({EDUC_COL})",
    f"C({INCOME_COL})",
    BMI_COL,
    f"C({SMOKING_COL})",
    f"C({SEASON_COL})",
]

COVARIATES_WITH_POST = ["post"] + COVARIATES_NO_POST

# ------------------------------------------------------------
# 6. Formatting helpers
# ------------------------------------------------------------

def format_pvalue(p):
    if pd.isna(p):
        return "—"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

def format_estimate_ci(beta, ci_low, ci_high):
    est = 100 * (np.exp(beta) - 1)
    low = 100 * (np.exp(ci_low) - 1)
    high = 100 * (np.exp(ci_high) - 1)
    return f"{est:+.1f}% [{low:+.1f}, {high:+.1f}]"

def apply_biomarker_order(tab):
    tab["biomarker"] = pd.Categorical(
        tab["biomarker"],
        categories=BIOMARKER_ORDER,
        ordered=True
    )
    return tab.sort_values(["lifestyle_factor", "biomarker"]).reset_index(drop=True)

In [ ]:
# ============================================================
# Table S5 — Associations of lifestyle-related factors with biomarkers
# ============================================================

table_s5_rows = []

for spec in LIFESTYLE_SPECS:
    group_var = spec["var"]
    group_label = spec["label"]
    reference = spec["reference"]
    
    cov_formula = " + ".join(COVARIATES_WITH_POST)
    
    for biomarker in BIOMARKERS:
        biomarker_label = BIOMARKER_LABELS[biomarker]
        
        required_cols = [
            biomarker, group_var, "post", AGE_COL, SEX_COL, EDUC_COL,
            INCOME_COL, BMI_COL, SMOKING_COL, SEASON_COL
        ]
        
        tmp = df_life[required_cols].copy()
        tmp = tmp.dropna(subset=required_cols)
        tmp = tmp[tmp[biomarker] > 0].copy()
        
        if tmp.empty:
            continue
        
        tmp["log_y"] = np.log(tmp[biomarker])
        
        formula = (
            f"log_y ~ C({group_var}, Treatment(reference='{reference}'))"
            f" + {cov_formula}"
        )
        
        try:
            fit = smf.ols(formula, data=tmp).fit()
            
            for param in fit.params.index:
                if f"C({group_var}" in param:
                    beta = fit.params[param]
                    ci_low, ci_high = fit.conf_int().loc[param]
                    p_value = fit.pvalues[param]
                    
                    # Extract level name from parameter
                    level = param.split("[T.")[-1].replace("]", "")
                    
                    table_s5_rows.append({
                        "lifestyle_factor": group_label,
                        "biomarker": biomarker_label,
                        "level": level,
                        "reference": reference,
                        "n": int(fit.nobs),
                        "estimate": format_estimate_ci(beta, ci_low, ci_high),
                        "estimate_percent": 100 * (np.exp(beta) - 1),
                        "ci_low_percent": 100 * (np.exp(ci_low) - 1),
                        "ci_high_percent": 100 * (np.exp(ci_high) - 1),
                        "p_value": p_value,
                        "error": None
                    })
                    
        except Exception as e:
            table_s5_rows.append({
                "lifestyle_factor": group_label,
                "biomarker": biomarker_label,
                "level": None,
                "reference": reference,
                "n": len(tmp),
                "estimate": None,
                "estimate_percent": np.nan,
                "ci_low_percent": np.nan,
                "ci_high_percent": np.nan,
                "p_value": np.nan,
                "error": str(e)
            })

table_s5 = pd.DataFrame(table_s5_rows)

# FDR correction across all lifestyle–biomarker contrasts
valid = table_s5["p_value"].notna()
table_s5["p_fdr"] = np.nan
if valid.sum() > 0:
    table_s5.loc[valid, "p_fdr"] = multipletests(
        table_s5.loc[valid, "p_value"],
        method="fdr_bh"
    )[1]

table_s5["p_value_fmt"] = table_s5["p_value"].apply(format_pvalue)
table_s5["p_fdr_fmt"] = table_s5["p_fdr"].apply(format_pvalue)

table_s5_display = table_s5[
    [
        "lifestyle_factor",
        "biomarker",
        "level",
        "reference",
        "n",
        "estimate",
        "p_value_fmt",
        "p_fdr_fmt",
        "error"
    ]
].copy()

table_s5_display = apply_biomarker_order(table_s5_display)

display(table_s5_display)

# Significant FDR results only
table_s5_significant = table_s5.loc[
    table_s5["p_fdr"] < 0.05
].sort_values("p_fdr")

display(table_s5_significant[
    [
        "lifestyle_factor", "biomarker", "level", "reference",
        "n", "estimate", "p_value", "p_fdr"
    ]
])

# Export
table_s5_display.to_excel(OUTPUT_DIR / "Table_S5_lifestyle_associations_display.xlsx", index=False)
table_s5.to_excel(OUTPUT_DIR / "Table_S5_lifestyle_associations_full.xlsx", index=False)
table_s5_significant.to_excel(OUTPUT_DIR / "Table_S5_lifestyle_associations_significant.xlsx", index=False)

In [ ]:
# ============================================================
# Table S6 — Stratified pre/post-pandemic differences
# by lifestyle-related categories
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def format_pvalue(p):
    if pd.isna(p):
        return "—"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def format_estimate_ci(beta, ci_low, ci_high):
    est = 100 * (np.exp(beta) - 1)
    low = 100 * (np.exp(ci_low) - 1)
    high = 100 * (np.exp(ci_high) - 1)
    return f"{est:+.1f}% [{low:+.1f}, {high:+.1f}]"


def get_interaction_pvalue(fit, group_var):
    """
    Global p-value for post × lifestyle group interaction.
    Uses all interaction terms involving post and the lifestyle factor.
    """
    params = fit.params.index.tolist()
    
    interaction_terms = [
        p for p in params
        if ("post:" in p or ":post" in p) and group_var in p
    ]
    
    if len(interaction_terms) == 0:
        return np.nan
    
    R = np.zeros((len(interaction_terms), len(params)))
    
    for i, term in enumerate(interaction_terms):
        R[i, params.index(term)] = 1
    
    try:
        test = fit.f_test(R)
        return float(test.pvalue)
    except Exception:
        return np.nan


def apply_biomarker_order(tab):
    tab = tab.copy()
    tab["biomarker"] = pd.Categorical(
        tab["biomarker"],
        categories=BIOMARKER_ORDER,
        ordered=True
    )
    return tab.sort_values(["lifestyle_factor", "biomarker", "level"]).reset_index(drop=True)


# ------------------------------------------------------------
# Main Table S6 loop
# ------------------------------------------------------------

table_s6_rows = []
interaction_rows = []

for spec in LIFESTYLE_SPECS:
    
    group_var = spec["var"]
    group_label = spec["label"]
    reference = spec["reference"]
    
    for biomarker in BIOMARKERS:
        
        biomarker_label = BIOMARKER_LABELS[biomarker]
        
        required_cols = [
            biomarker,
            group_var,
            "post",
            AGE_COL,
            SEX_COL,
            EDUC_COL,
            INCOME_COL,
            BMI_COL,
            SMOKING_COL,
            SEASON_COL
        ]
        
        tmp = df_life[required_cols].copy()
        tmp = tmp.dropna(subset=required_cols)
        tmp = tmp[tmp[biomarker] > 0].copy()
        
        if tmp.empty:
            interaction_rows.append({
                "lifestyle_factor": group_label,
                "biomarker": biomarker_label,
                "p_interaction": np.nan,
                "n_model": 0,
                "error": "Empty dataset after exclusions"
            })
            continue
        
        if tmp["post"].nunique() < 2 or tmp[group_var].nunique() < 2:
            interaction_rows.append({
                "lifestyle_factor": group_label,
                "biomarker": biomarker_label,
                "p_interaction": np.nan,
                "n_model": len(tmp),
                "error": "Not enough post/pre or subgroup variation"
            })
            continue
        
        tmp["log_y"] = np.log(tmp[biomarker])
        
        # Ensure categorical group and reference category
        tmp[group_var] = tmp[group_var].astype("category")
        
        if reference not in list(tmp[group_var].cat.categories):
            interaction_rows.append({
                "lifestyle_factor": group_label,
                "biomarker": biomarker_label,
                "p_interaction": np.nan,
                "n_model": len(tmp),
                "error": f"Reference category not found: {reference}"
            })
            continue
        
        # ----------------------------------------------------
        # 1. Global interaction model for p-interaction
        # ----------------------------------------------------
        
        interaction_formula = (
            f"log_y ~ post * C({group_var}, Treatment(reference='{reference}'))"
            f" + {AGE_COL}"
            f" + C({SEX_COL})"
            f" + C({EDUC_COL})"
            f" + C({INCOME_COL})"
            f" + {BMI_COL}"
            f" + C({SMOKING_COL})"
            f" + C({SEASON_COL})"
        )
        
        try:
            fit_interaction = smf.ols(interaction_formula, data=tmp).fit()
            p_interaction = get_interaction_pvalue(fit_interaction, group_var)
            interaction_error = None
        except Exception as e:
            p_interaction = np.nan
            interaction_error = str(e)
        
        interaction_rows.append({
            "lifestyle_factor": group_label,
            "biomarker": biomarker_label,
            "p_interaction": p_interaction,
            "n_model": len(tmp),
            "error": interaction_error
        })
        
        # ----------------------------------------------------
        # 2. Stratified post-vs-pre estimates
        # ----------------------------------------------------
        
        levels = list(tmp[group_var].cat.categories)
        levels = [lev for lev in levels if lev in list(tmp[group_var].dropna().unique())]
        
        for level in levels:
            
            d_level = tmp.loc[tmp[group_var] == level].copy()
            
            n_pre = int((d_level["post"] == 0).sum())
            n_post = int((d_level["post"] == 1).sum())
            
            if d_level["post"].nunique() < 2:
                table_s6_rows.append({
                    "lifestyle_factor": group_label,
                    "biomarker": biomarker_label,
                    "level": level,
                    "n_pre": n_pre,
                    "n_post": n_post,
                    "post_vs_pre": None,
                    "post_vs_pre_percent": np.nan,
                    "ci_low_percent": np.nan,
                    "ci_high_percent": np.nan,
                    "p_post": np.nan,
                    "p_interaction": p_interaction,
                    "error": "No pre/post variation within stratum"
                })
                continue
            
            stratified_formula = (
                f"log_y ~ post"
                f" + {AGE_COL}"
                f" + C({SEX_COL})"
                f" + C({EDUC_COL})"
                f" + C({INCOME_COL})"
                f" + {BMI_COL}"
                f" + C({SMOKING_COL})"
                f" + C({SEASON_COL})"
            )
            
            try:
                fit_level = smf.ols(stratified_formula, data=d_level).fit()
                
                beta = fit_level.params["post"]
                ci_low, ci_high = fit_level.conf_int().loc["post"]
                p_post = fit_level.pvalues["post"]
                
                table_s6_rows.append({
                    "lifestyle_factor": group_label,
                    "biomarker": biomarker_label,
                    "level": level,
                    "n_pre": n_pre,
                    "n_post": n_post,
                    "post_vs_pre": format_estimate_ci(beta, ci_low, ci_high),
                    "post_vs_pre_percent": 100 * (np.exp(beta) - 1),
                    "ci_low_percent": 100 * (np.exp(ci_low) - 1),
                    "ci_high_percent": 100 * (np.exp(ci_high) - 1),
                    "p_post": p_post,
                    "p_interaction": p_interaction,
                    "error": None
                })
                
            except Exception as e:
                table_s6_rows.append({
                    "lifestyle_factor": group_label,
                    "biomarker": biomarker_label,
                    "level": level,
                    "n_pre": n_pre,
                    "n_post": n_post,
                    "post_vs_pre": None,
                    "post_vs_pre_percent": np.nan,
                    "ci_low_percent": np.nan,
                    "ci_high_percent": np.nan,
                    "p_post": np.nan,
                    "p_interaction": p_interaction,
                    "error": str(e)
                })


# ------------------------------------------------------------
# Build dataframes
# ------------------------------------------------------------

table_s6 = pd.DataFrame(table_s6_rows)
table_s6_interactions = pd.DataFrame(interaction_rows)

print("Rows in Table S6:", len(table_s6))
print("Rows in interaction table:", len(table_s6_interactions))

# ------------------------------------------------------------
# FDR correction for interaction p-values
# ------------------------------------------------------------

table_s6_interactions["p_interaction_fdr"] = np.nan

valid = table_s6_interactions["p_interaction"].notna()

if valid.sum() > 0:
    table_s6_interactions.loc[valid, "p_interaction_fdr"] = multipletests(
        table_s6_interactions.loc[valid, "p_interaction"],
        method="fdr_bh"
    )[1]

# ------------------------------------------------------------
# Merge FDR p-values into Table S6
# ------------------------------------------------------------

table_s6 = table_s6.merge(
    table_s6_interactions[
        ["lifestyle_factor", "biomarker", "p_interaction_fdr", "n_model"]
    ],
    on=["lifestyle_factor", "biomarker"],
    how="left"
)

table_s6["p_post_fmt"] = table_s6["p_post"].apply(format_pvalue)
table_s6["p_interaction_fmt"] = table_s6["p_interaction"].apply(format_pvalue)
table_s6["p_interaction_fdr_fmt"] = table_s6["p_interaction_fdr"].apply(format_pvalue)

# ------------------------------------------------------------
# Display table
# ------------------------------------------------------------

table_s6_display = table_s6[
    [
        "lifestyle_factor",
        "biomarker",
        "level",
        "n_pre",
        "n_post",
        "post_vs_pre",
        "p_post_fmt",
        "p_interaction_fmt",
        "p_interaction_fdr_fmt",
        "error"
    ]
].copy()

table_s6_display = apply_biomarker_order(table_s6_display)

display(table_s6_display)

# ------------------------------------------------------------
# Significant FDR interactions
# ------------------------------------------------------------

table_s6_significant_interactions = table_s6_interactions.loc[
    table_s6_interactions["p_interaction_fdr"] < 0.05
].sort_values("p_interaction_fdr")

display(table_s6_significant_interactions)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

table_s6_display.to_excel(
    OUTPUT_DIR / "Table_S6_lifestyle_stratified_prepost_display.xlsx",
    index=False
)

table_s6.to_excel(
    OUTPUT_DIR / "Table_S6_lifestyle_stratified_prepost_full.xlsx",
    index=False
)

table_s6_interactions.to_excel(
    OUTPUT_DIR / "Table_S6_lifestyle_interactions.xlsx",
    index=False
)

table_s6_significant_interactions.to_excel(
    OUTPUT_DIR / "Table_S6_lifestyle_interactions_significant.xlsx",
    index=False
)

In [ ]:
BIOMARKER_ORDER_LABELS = ["CRP", "MCP-1", "TNF-α", "IFNγ", "eotaxin", "IL-6", "IL-8", "IL-10", "sICAM-1", "sVCAM-1", "VEGF-A", "MMP-9"]
LIFESTYLE_ORDER = ["Physical activity", "Sedentary time", "Sleep duration", "Alcohol intake"]
LEVEL_ORDER = {
    "Physical activity": ["Low physical activity", "Medium physical activity", "High physical activity"],
    "Sedentary time": ["Low sedentary time", "Medium sedentary time", "High sedentary time"],
    "Sleep duration": ["Short sleep", "Normal sleep", "Long sleep"],
    "Alcohol intake": ["Low alcohol intake", "Medium alcohol intake", "High alcohol intake"],
}

def order_lifestyle_table(tab):
    tab = tab.copy()
    tab["biomarker"] = pd.Categorical(tab["biomarker"], BIOMARKER_ORDER_LABELS, ordered=True)
    tab["lifestyle_factor"] = pd.Categorical(tab["lifestyle_factor"], LIFESTYLE_ORDER, ordered=True)
    tab["_level_order"] = tab.apply(lambda r: LEVEL_ORDER.get(str(r["lifestyle_factor"]), []).index(str(r["level"])) if str(r["level"]) in LEVEL_ORDER.get(str(r["lifestyle_factor"]), []) else 999, axis=1)
    return tab.sort_values(["biomarker", "lifestyle_factor", "_level_order"]).drop(columns="_level_order").reset_index(drop=True)

table_s5_grouped = order_lifestyle_table(table_s5_display)[[
    "biomarker", "lifestyle_factor", "level", "reference", "n", "estimate", "p_value_fmt", "p_fdr_fmt"
]]
table_s6_grouped = order_lifestyle_table(table_s6_display)[[
    "biomarker", "lifestyle_factor", "level", "n_pre", "n_post", "post_vs_pre", "p_post_fmt", "p_interaction_fmt", "p_interaction_fdr_fmt"
]]

with pd.ExcelWriter(RESULTS_DIR / "Tables_S5_S6_lifestyle.xlsx") as writer:
    table_s5_grouped.to_excel(writer, sheet_name="Table_S5", index=False)
    table_s6_grouped.to_excel(writer, sheet_name="Table_S6", index=False)
print("Saved:", RESULTS_DIR / "Tables_S5_S6_lifestyle.xlsx")

## 10. Figure 3 — Lifestyle-stratified pre/post-pandemic differences

In [ ]:
from matplotlib.lines import Line2D

plot_df = table_s6.copy()
if "error" in plot_df.columns:
    plot_df = plot_df[plot_df["error"].isna()].copy()

plot_df["biomarker"] = plot_df["biomarker"].replace({"eotaxin": "Eotaxin"})

def simplify_level(x):
    txt = str(x).lower()
    for key, out in [("low", "Low"), ("medium", "Medium"), ("high", "High"), ("short", "Short"), ("normal", "Normal"), ("long", "Long")]:
        if key in txt:
            return out
    return str(x)

plot_df["level_simple"] = plot_df["level"].apply(simplify_level)

BIOMARKER_ORDER_FIG3 = ["CRP", "MCP-1", "TNF-α", "IFNγ", "Eotaxin", "IL-6", "IL-8", "IL-10", "sICAM-1", "sVCAM-1", "VEGF-A", "MMP-9"]
LIFESTYLE_ORDER_FIG3 = ["Physical activity", "Sedentary time", "Sleep duration", "Alcohol intake"]
LEVEL_ORDER_FIG3 = {
    "Physical activity": ["Low", "Medium", "High"],
    "Sedentary time": ["Low", "Medium", "High"],
    "Sleep duration": ["Short", "Normal", "Long"],
    "Alcohol intake": ["Low", "Medium", "High"],
}
STYLE_BY_RANK = {
    0: {"color": "#1f77b4", "marker": "o"},
    1: {"color": "#ff7f0e", "marker": "s"},
    2: {"color": "#2ca02c", "marker": "^"},
}
offsets = {0: -0.22, 1: 0.0, 2: 0.22}

for col in ["post_vs_pre_percent", "ci_low_percent", "ci_high_percent", "p_interaction_fdr"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")
plot_df = plot_df.dropna(subset=["post_vs_pre_percent", "ci_low_percent", "ci_high_percent"])

xmin = np.floor(plot_df["ci_low_percent"].min() / 10) * 10 - 2
xmax = np.ceil(plot_df["ci_high_percent"].max() / 10) * 10 + 8

y_lookup = {b: len(BIOMARKER_ORDER_FIG3) - 1 - i for i, b in enumerate(BIOMARKER_ORDER_FIG3)}
fig, axes = plt.subplots(2, 2, figsize=(20, 14), sharex=True, sharey=True)
axes = axes.flatten()

for ax, lifestyle in zip(axes, LIFESTYLE_ORDER_FIG3):
    sub = plot_df[plot_df["lifestyle_factor"] == lifestyle].copy()
    handles = []
    for i, level in enumerate(LEVEL_ORDER_FIG3[lifestyle]):
        style = STYLE_BY_RANK[i]
        handles.append(Line2D([0], [0], marker=style["marker"], color=style["color"], linestyle="none", markersize=7, label=level))
        d = sub[sub["level_simple"] == level]
        for _, row in d.iterrows():
            b = str(row["biomarker"])
            if b not in y_lookup:
                continue
            y = y_lookup[b] + offsets[i]
            x, lo, hi = row["post_vs_pre_percent"], row["ci_low_percent"], row["ci_high_percent"]
            ax.errorbar(x, y, xerr=[[x-lo], [hi-x]], fmt=style["marker"], color=style["color"], ecolor=style["color"], capsize=3, elinewidth=1.4, markersize=7, alpha=0.9)

    ax.axvline(0, linestyle="--", linewidth=1.2, color="black")
    ax.grid(axis="x", alpha=0.25)
    ax.set_title(lifestyle, fontsize=17, fontweight="bold")
    ax.set_yticks([y_lookup[b] for b in BIOMARKER_ORDER_FIG3])
    ax.set_yticklabels(BIOMARKER_ORDER_FIG3, fontsize=13)
    ax.set_xlim(xmin, xmax)

    # Annotate only interactions with FDR-adjusted P < 0.05, using neutral numerical wording.
    sig = sub[["biomarker", "p_interaction_fdr"]].drop_duplicates().dropna()
    sig = sig[sig["p_interaction_fdr"] < 0.05]
    for _, row in sig.iterrows():
        b = str(row["biomarker"])
        if b in y_lookup:
            ax.text(xmax - 0.5, y_lookup[b], f"FDR-adjusted P={row['p_interaction_fdr']:.3f}", va="center", ha="right", fontsize=10, fontweight="bold")

    ax.legend(handles=handles, title="Level", frameon=False, loc="lower right", fontsize=11, title_fontsize=11)

fig.supxlabel("Adjusted percentage difference: post-pandemic vs pre-pandemic (%)", fontsize=15, y=0.04)
fig.supylabel("Inflammatory biomarkers", fontsize=15, x=0.04)
plt.tight_layout(rect=[0.04, 0.06, 1, 0.98])
fig.savefig(FIGURES_DIR / "Figure3_lifestyle_stratified_prepost.png", dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(FIGURES_DIR / "Figure3_lifestyle_stratified_prepost.pdf", bbox_inches="tight", facecolor="white")
plt.show()